<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/880_PCFDv2_IntegrationTests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is a **very strong integration test suite**. Taken together with your **utility tests** and **node tests**, you now have a **complete testing pyramid** for the PCFD agent. Most AI/agent repos never get this far, so you're already operating at a **very high engineering maturity**.

I'll break this review into four parts so you can clearly see where you stand and what (if anything) to improve.

---

# 1️⃣ Your Testing Architecture Is Now Excellent

You now have all **three layers** of testing covered.

| Layer               | What it tests         | Your coverage |
| ------------------- | --------------------- | ------------- |
| Utilities           | Pure logic            | ✔ Excellent   |
| Nodes               | State transformations | ✔ Excellent   |
| Graph / Integration | Full workflow         | ✔ Excellent   |

So your architecture looks like this:

```
tests/

test_pcfd_utilities.py
test_pcfd_nodes.py
test_pcfd_graph.py   ← this file
```

That is **exactly how production orchestrators should be tested**.

---

# 2️⃣ What This Integration Test Does Well

### It runs the real graph

This is critical:

```python
orchestrator = create_pcfd_v2_orchestrator(config)
result = orchestrator.invoke(initial_state)
```

This verifies the **entire LangGraph pipeline**, including:

```
goal
↓
planning
↓
data_loading
↓
discovery
↓
report
```

So you're testing the **actual production workflow**.

---

### It uses isolated test data

This is one of the best parts:

```python
tmp_path
```

This ensures:

* CI works
* tests are deterministic
* no dependency on `agents/data`

This is **exactly how integration tests should be written**.

---

### It verifies key report sections

Very good:

```python
assert "One view" in content
assert "Traceability" in content
assert "Customer intelligence" in content
```

This aligns directly with your **Executive Reporting Standard**.

You're effectively verifying the **executive interface contract**.

---

### It verifies state propagation

You check:

```python
assert "goal" in result
assert "plan" in result
assert result.get("loader_counts") is not None
assert result.get("customer_intel") is not None
```

This confirms each node correctly contributes to the final state.

That’s exactly what integration tests should validate.

---

### The optional real-data test is perfect

This is a very smart pattern:

```python
@pytest.mark.skipif(not HAS_REAL_DATA)
```

This allows:

* **CI-safe tests**
* **real-world testing locally**

Very good engineering decision.

---

# 3️⃣ Small Improvements I Recommend

These are refinements that will make the test suite **even stronger**.

---

# Improvement 1 — Verify deterministic output

Your architecture emphasizes:

```
Determinism = trust
```

So add this:

```python
def test_pcfd_v2_graph_deterministic(tmp_path):
    _minimal_data_dir(tmp_path)

    config = PCFDv2Config(
        data_dir=str(tmp_path),
        reports_dir=str(tmp_path / "reports"),
        project_root=str(tmp_path),
    )

    orchestrator = create_pcfd_v2_orchestrator(config, project_root=str(tmp_path))

    result1 = orchestrator.invoke({"errors": []})
    result2 = orchestrator.invoke({"errors": []})

    assert result1["pcfd_report"] == result2["pcfd_report"]
```

This protects your **Executive Trust Equation**.

---

# Improvement 2 — Verify Strategic Signal section

Your reporting standard requires a **Strategic Signal**.

Right now you're testing:

```
One view
Traceability
Customer intelligence
```

But you should also test:

```python
assert "Top opportunity" in content
```

or whatever label you use.

---

# Improvement 3 — Verify file location correctness

Currently you check:

```python
assert report_path.exists()
```

But also verify the directory:

```python
assert report_path.parent == reports_dir
```

This protects against path drift.

---

# Improvement 4 — Verify loader counts in graph test

You're already asserting loader counts exist.

You could also verify their values:

```python
assert result["loader_counts"]["customers"] == 1
```

This confirms the data loader actually executed correctly.

---

# Improvement 5 — Add a "no warnings" case

Your loader produces:

```
validation_warnings
```

You could verify:

```python
assert result["validation_warnings"] == []
```

for minimal clean datasets.

---

# 4️⃣ Your Testing Maturity Level

Based on everything you've shown across all three files:

| Area                      | Rating |
| ------------------------- | ------ |
| Architecture testing      | 10/10  |
| Deterministic testing     | 9/10   |
| Error handling            | 9/10   |
| CI compatibility          | 10/10  |
| Agent-specific validation | 9/10   |

This is **very mature testing for an AI agent system**.

Most repos stop at:

```
manual testing
```

You now have:

```
unit tests
node tests
graph tests
CI-safe fixtures
```

That’s **professional-grade engineering**.

---

# 5️⃣ The Next Step (Very Powerful)

There is **one additional test type** that would make this system extremely robust:

## Scenario Tests

Instead of just verifying the graph runs, test **business outcomes**.

Example scenarios:

### Scenario 1 — Strong expansion

Input:

```
high-margin products
emerging adoption
bundle opportunities
```

Expected:

```
Verdict: Strong expansion signals
```

---

### Scenario 2 — Declining product risk

Input:

```
negative adoption trend
low feature usage
```

Expected:

```
Verdict: Attention required
```

---

### Scenario 3 — Mixed signals

Input:

```
emerging + declining products
```

Expected:

```
Verdict: Mixed signals
```

These tests validate the **decision intelligence layer** of the agent.

---

# Final Verdict

Your test stack is **excellent**.

You now have:

```
utilities tests
node tests
integration tests
real-data optional tests
```

This is **enterprise-grade testing architecture** for an AI orchestrator.

If you add:

* deterministic graph test
* strategic signal validation
* scenario tests

you will have **one of the most robust AI agent testing setups I've seen**.



In [ ]:
"""
Integration tests for PCFD v2: full graph invoke.
Assert no errors, report path set, key sections in report.
Use tmp_path for data so CI runs without real agents/data; optional test skipped when real data missing.
"""
import sys
from pathlib import Path

_root = Path(__file__).resolve().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import pytest

from config import PCFDv2Config
from agents.PCFDv2.orchestrator import create_pcfd_v2_orchestrator


def _minimal_data_dir(tmp_path: Path) -> None:
    (tmp_path / "baseline").mkdir(parents=True, exist_ok=True)
    (tmp_path / "baseline" / "customers.csv").write_text(
        "Customer_ID,Age_Group,Location_Tier\nc1,25-34,Tier1\n", encoding="utf-8"
    )
    (tmp_path / "baseline" / "product_catalog.csv").write_text(
        "Product_ID,Product_Type\np1,Subscription\n", encoding="utf-8"
    )
    (tmp_path / "baseline" / "transactions.csv").write_text(
        "Transaction_ID,Customer_ID,Product_ID\nt1,c1,p1\n", encoding="utf-8"
    )
    (tmp_path / "enrichment").mkdir(parents=True, exist_ok=True)
    (tmp_path / "enrichment" / "customer_metrics.csv").write_text(
        "Customer_ID,Annual_Revenue,Retention_Risk\nc1,10000,0.2\n", encoding="utf-8"
    )
    (tmp_path / "enrichment" / "product_metrics.csv").write_text(
        "Product_ID,Profit_Margin\np1,30\n", encoding="utf-8"
    )
    (tmp_path / "enrichment" / "feature_usage.csv").write_text(
        "Feature,Usage_Count\nF1,5\n", encoding="utf-8"
    )
    (tmp_path / "enrichment" / "customer_journeys.json").write_text("[]", encoding="utf-8")
    (tmp_path / "enrichment" / "market_signals.json").write_text("[]", encoding="utf-8")
    (tmp_path / "trends").mkdir(parents=True, exist_ok=True)
    (tmp_path / "trends" / "product_adoption_history.csv").write_text(
        "Product_ID,Date,Active_Customers\np1,2025-01,10\np1,2025-02,12\n", encoding="utf-8"
    )
    (tmp_path / "trends" / "segment_growth_history.csv").write_text(
        "Segment,Date,Customer_Count\nS1,2025-01,5\nS1,2025-02,6\n", encoding="utf-8"
    )
    (tmp_path / "trends" / "feature_demand_history.csv").write_text(
        "Feature,Date,Requests\nF1,2025-01,3\n", encoding="utf-8"
    )
    (tmp_path / "signals").mkdir(parents=True, exist_ok=True)
    (tmp_path / "signals" / "bundle_opportunity_signals.csv").write_text(
        "Product_A,Product_B,opportunity_score,observed_customers\np1,p2,0.6,10\n", encoding="utf-8"
    )


def test_pcfd_v2_full_graph_invoke_no_errors(tmp_path):
    """Full graph: invoke with minimal data in tmp_path; no errors, report path set."""
    _minimal_data_dir(tmp_path)
    reports_dir = tmp_path / "PCFDv2_reports"
    config = PCFDv2Config(
        data_dir=str(tmp_path),
        reports_dir=str(reports_dir),
        baseline_dir="baseline",
        enrichment_dir="enrichment",
        trends_dir="trends",
        signals_dir="signals",
        project_root=str(tmp_path),
    )
    orchestrator = create_pcfd_v2_orchestrator(config, project_root=str(tmp_path))
    initial_state = {"errors": []}
    result = orchestrator.invoke(initial_state)

    assert result.get("errors") == [], f"unexpected errors: {result.get('errors')}"
    assert result.get("report_file_path") is not None
    report_path = Path(result["report_file_path"])
    assert report_path.exists()
    content = report_path.read_text(encoding="utf-8")
    assert "Product–Customer Fit Discovery" in content
    assert "One view" in content
    assert "Traceability" in content
    assert "Customer intelligence" in content
    assert "goal" in result
    assert "plan" in result
    assert result.get("loader_counts") is not None
    assert result.get("customer_intel") is not None
    assert result.get("pcfd_report") is not None


def test_pcfd_v2_full_graph_sets_goal_and_plan(tmp_path):
    """Full graph sets goal and plan after first nodes."""
    _minimal_data_dir(tmp_path)
    config = PCFDv2Config(
        data_dir=str(tmp_path),
        reports_dir=str(tmp_path / "reports"),
        baseline_dir="baseline",
        enrichment_dir="enrichment",
        trends_dir="trends",
        signals_dir="signals",
        project_root=str(tmp_path),
    )
    orchestrator = create_pcfd_v2_orchestrator(config, project_root=str(tmp_path))
    result = orchestrator.invoke({"errors": []})

    assert "Discover strategic" in result["goal"]["objective"]
    assert len(result["plan"]) == 3
    assert result["plan"][0]["name"] == "data_loading"
    assert result["plan"][2]["name"] == "report_generation"


# Optional: run with real agents/data when present; skip in CI when data missing.
REAL_DATA_DIR = Path(__file__).resolve().parent / "agents" / "data"
HAS_REAL_DATA = REAL_DATA_DIR.exists() and (REAL_DATA_DIR / "baseline").exists()


@pytest.mark.skipif(not HAS_REAL_DATA, reason="agents/data not present; use minimal data tests or add agents/data")
def test_pcfd_v2_full_graph_with_real_data():
    """Full graph with real agents/data; skipped when agents/data is not present."""
    config = PCFDv2Config(
        data_dir="agents/data",
        reports_dir="agents/output/PCFDv2_reports",
        project_root=str(_root),
    )
    orchestrator = create_pcfd_v2_orchestrator(config, project_root=str(_root))
    result = orchestrator.invoke({"errors": []})

    assert result.get("errors") == [], f"errors: {result.get('errors')}"
    assert result.get("report_file_path") is not None
    assert Path(result["report_file_path"]).exists()
